# 1. Load and convert labeled data into sequential datasets

path/to/labeled_data/ 

    ├── class_0/
    │   ├── sample1.pkl
    │   ├── sample2.csv
    │   └── ...
    ├── class_1/
    │   ├── sample1.csv
    │   ├── sample2.pkl
    │   └── ...
    └── ...

dict_label={'class_0':0, 'class_1':1, ... }

In [17]:
import numpy as np
import os
import pandas as pd
from torch.utils.data import Dataset
import torch
from torchinfo import summary 
import random
from torch.utils.data import random_split, DataLoader, Subset, ConcatDataset


class LoadSeqDataset(Dataset):
    def __init__(self, file_path: str, label: int, selected_features:list, seq_num=28, gap=5, desire_class = None):
        """
        Initialize the dataset from a labeled data file by converting it to sequential format.
        
        Args:
            file_path (str): Path to the labeled data file.
            label (int): Label associated with the sequences from this file.
            seq_num (int): Length of each sequence.
            gap (int): Step size between sequences.
        """
        self.seq_num = seq_num
        self.gap = gap
        self.selected_features = selected_features
        # Define dtype for selected features and label
        dtypes = {col: 'float32' for col in selected_features}
        dtypes['label'] = 'int8'  # Set label as an integer type
        dtypes['time'] = 'float32'  # Set label as an integer type
        # Load data and process into sequences

        self.data = pd.read_csv(file_path, usecols=selected_features + ['label', 'time'], dtype = dtypes)
        self.data.drop(columns='index', inplace=True, errors='ignore')
        self.data.label = self.data.label * label.item()
        self.df = self.data
        #self.sequences = self._make_sequences_contact_only()
        self.sequences = self._make_sequences()
        if desire_class is not None:
            self.sequences = self._get_specific_class(desire_class)
    def balanceData(self, split_rate_1 = 1, split_rate_0=0.15):
        """Balances the dataset by downsampling sequences where label = 0 to 10%."""
        # Separate sequences based on the label
        label_0_sequences = [seq for seq in self.sequences if seq[1] == 0]
        other_label_sequences = [seq for seq in self.sequences if seq[1] != 0]

        # Downsample label 0 sequences to 10%
        num_to_keep = int(len(label_0_sequences) * split_rate_0)
        downsampled_label_0 = random.sample(label_0_sequences, num_to_keep)

        num_to_keep = int(len(other_label_sequences) * split_rate_1)
        other_label_sequences = random.sample(other_label_sequences, num_to_keep)

        # Combine and shuffle
        balanced_sequences = downsampled_label_0 + other_label_sequences
        random.shuffle(balanced_sequences)

        # Update sequences
        self.sequences = balanced_sequences

    def _make_sequences_contact_only(self):
        """Generate sequences based on the contact points detected in the data."""
        start_contact_indexs = self.df.loc[self.df.label.diff() > 0.1, :].index
        end_contact_indexs = self.df.loc[self.df.label.diff() < -0.1, :].index - 1
        contact_indexs = [idx for idx, idx2 in zip(start_contact_indexs, end_contact_indexs) if idx2 - idx >= self.seq_num]

        sequences = []
        for contact_index in contact_indexs:
            end_point = contact_index + self.seq_num
            for step in range(contact_index, end_point, self.gap):
                window = self.df[self.selected_features][step - self.seq_num + 1:step + 1]
                sequences.append((window.values, self.df.label[step]))
        return sequences
    
    def _make_sequences(self):
        """Generate sequences over time"""
        sequences = []
        for step in range(self.seq_num,self.data.shape[0], self.gap):
            window = self.df[self.selected_features][step - self.seq_num:step]
            sequences.append((window.values, self.df.label[step-1]))
        
        return sequences

    def _get_specific_class(self, desired_label):
        filtered_data = [(seq, label) for seq, label in self.sequences if label == desired_label]
        return filtered_data


    def __len__(self):
        """Return the total number of sequences in the dataset."""
        return len(self.sequences)

    def __getitem__(self, idx):
        """
        Retrieve a single sequence and label.
        
        Args:
            idx (int): Index of the sequence to retrieve.
            
        Returns:
            (tuple): (features, target) where target is the label for classification.
        """
        #TODO: multiple features should be reshaped.
        features, target = self.sequences[idx]
        features = features.T if isinstance(features, torch.Tensor) else torch.tensor(features, dtype=torch.float32).T.clone().detach()
        target = target if isinstance(target, torch.Tensor) else torch.tensor(target, dtype=torch.long).clone().detach()
        return features, target


class LoadDatasets(Dataset):
    def __init__(self, data_path:str, dict_label = None):
        """
        Load sequential dataset from a directory structure with labeled subdirectories.

            Expected directory structure:
            
            path/to/data/
                ├── class_0/
                │   ├── sample1.pkl 
                │   ├── sample2.csv
                │   └── ...
                ├── class_1/
                │   ├── sample1.csv
                │   ├── sample2.pkl
                │   └── ...
                └── ...

            Label mapping example:
            
            dict_label = {'class_0': 0, 'class_1': 1, ... }
        
        Args:
            data_path (str): Path to the data directory.
            dict_label (dict, optional): Dictionary mapping class folder names to labels.
        """
        if dict_label is None:
            dict_label = {'a': 7, 'b': 6, 'c': 5, 'd': 4, 'e': 3, 'f': 2, 'g': 1}
            #dict_label = {'link7': 7, 'link6':6, 'link5':5, 'link4':4, 'link3':3, 'link2':2, 'link1':1}

        
        self.samples = []
        self.class_to_idx = {}

        # Scan data_path for subdirectories        
        for class_name in sorted(os.listdir(data_path)):
            class_dir = os.path.join(data_path, class_name)
            if os.path.isdir(class_dir) and class_name in dict_label:
                label = dict_label[class_name]  # Look up label
                self.class_to_idx[class_name] = label
                for file_name in os.listdir(class_dir):
                    file_path = os.path.join(class_dir, file_name)
                    if os.path.isfile(file_path):
                        self.samples.append((file_path, label))
                        
    def __len__(self):
        """Return the total number of samples."""
        return len(self.samples)

    def __getitem__(self, idx):
        """
        Retrieve a single sample at the specified index.
        
        Args:
            idx (int): Index of the sample to retrieve.
            
        Returns:
            tuple: (file_path, label)
        """
        seq_path, label = self.samples[idx]
        return seq_path, label

# 2. load AI models

In [18]:
import argparse
import os
import time
import torch
import numpy as np
import random
import torch.optim as optim
import torch.nn as nn
from torchmetrics import ConfusionMatrix, Accuracy
import matplotlib.pyplot as plt
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

   

class cnnLSTM(nn.Module):
    def __init__(self, num_features_joints=28, hidden_size=32, num_layers=3, dropout=0.5, bidirectional=False):
        super(cnnLSTM, self).__init__()
        self.normalization = nn.LayerNorm(num_features_joints)#ZScoreNormalization() #nn.LayerNorm(num_features_joints)#
        
         # Define the 1D CNN layers
        self.cnn1 = nn.Conv1d(in_channels=num_features_joints, out_channels=256, kernel_size=1, padding='same',padding_mode='circular')
        self.cnn2 = nn.Conv1d(in_channels=256, out_channels=512, kernel_size=2, padding='same',padding_mode='circular')
        self.cnn3 = nn.Conv1d(in_channels=512, out_channels=256, kernel_size=4, padding='same',padding_mode='circular')
        
        self.bn1 = nn.BatchNorm1d(256)
        self.bn2 = nn.BatchNorm1d(512)
        self.bn3 = nn.BatchNorm1d(256)
        self.relu = nn.LeakyReLU() #nn.ReLU()
        self.pool = nn.MaxPool1d(kernel_size=2)
        
        # Define the LSTM layer
        self.lstm = nn.LSTM(
            input_size=256//2,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=dropout if num_layers > 1 else 0  # Dropout only if num_layers > 1
        )

        lstm_output_size = hidden_size * 2 if bidirectional else hidden_size
        self.normalization_lstm = nn.LayerNorm(lstm_output_size)
        
        # Fully connected layer
        self.fc = nn.Linear(lstm_output_size, 1)

    
        # Dropout for regularization
        self.dropout = nn.Dropout(dropout)
        
        self.lstm_out_calculation = nn.AdaptiveAvgPool1d(1)

    def forward(self, input):
        # Normalize input (batch_size, sequence_length, input_size)
        self.lstm.flatten_parameters()

        normalized_input = self.normalization(input)

        # Reshape for CNN: (batch_size, in_channels, sequence_length)
        cnn_input = normalized_input.permute(0, 2, 1)
        
        # Apply CNN layers
        cnn_out = self.relu(self.bn1(self.cnn1(cnn_input)))#, seq_pose = 1))
        cnn_out = self.relu(self.bn2(self.cnn2(cnn_out)))#, seq_pose = 1))
        cnn_out = self.relu(self.bn3(self.cnn3(cnn_out)))#, seq_pose = 1))
        
        cnn_out = cnn_out.permute(0,2,1)
        cnn_out = self.pool(cnn_out)

        # Reshape for LSTM: (batch_size, sequence_l98pength, input_size)

        lstm_out, _ = self.lstm(cnn_out)
        lstm_out = self.normalization_lstm(lstm_out)
        # Apply dropout
        lstm_out = self.dropout(lstm_out)
        lstm_out = self.lstm_out_calculation(lstm_out.permute(0, 2, 1)).squeeze(-1)
        # Fully connected layer
        
        joint_step_outputs = self.fc(lstm_out)  # (batch_size, 1, 1)

        # Squeeze to remove the last dimension
        joint_step_outputs = joint_step_outputs.squeeze(-1)  # Shape: (batch_size, 1)
        return joint_step_outputs
    
    def prediction(self, input):
        device = input.device
        output = self.forward(input)
        return (output > 0.5).int()
        #return torch.argmax(output, dim=1)

# 3. Training

In [19]:
# functions
import copy
import plotly.express as px
import chart_studio.plotly as py
import plotly.graph_objs as go
from plotly.offline import iplot, init_notebook_mode
# Using plotly + cufflinks in offline mode
import cufflinks
cufflinks.go_offline(connected=True)
init_notebook_mode(connected=True)
from collections import Counter

def create_hierarchical_labels(contact_indices, num_links):
    """
    Generate hierarchical labels for a batch of contact indices.
    
    Args:
        contact_indices (Tensor): A tensor of shape [batch_size] with the contact index for each example in the batch.
        num_links (int): The number of links (length of the output vector).
    
    Returns:
        Tensor: A tensor of shape [batch_size, num_links] containing hierarchical labels for each example.
    """
    batch_size = contact_indices.size(0)  # Get batch size
    y_true = torch.zeros(batch_size, dtype=torch.float32, device=contact_indices.device)  # Initialize a tensor of zeros
    
    for i in range(batch_size):
        contact_index = contact_indices[i]
        if contact_index == 0:
            y_true[i] = 0
        else:
            y_true[i] = 1 
    return y_true
'''
    contact_indices[(contact_indices >0)]=1
    return contact_indices #y_true'''
# Function to calculate detection delay

def contact_detection_accuracy(df, allowable_delay = 30):
    "df should include label, majority voting and time"
    TP, TN, FP, FN = 0, 0, 0, 0
    
    df['label_diff'] = df['label'].diff().fillna(0)
    df['TP'], df['TN'], df['FP'], df['FN'] = None, None, None, None

    contact_delays = []
    no_contact_delays = []
    df = df.reset_index(drop=True)

    change_events = df[df['label_diff'] != 0].index.tolist()

    # Add first and last index if not present
    if 0 not in change_events:
        change_events.insert(0, 0)
    if len(df) - 1 not in change_events:
        change_events.append(len(df) - 1)

    for event_idx in range(len(change_events) - 1):
        idx = change_events[event_idx]
        
        if df.label[idx] == 1: # Starting a contact event
            for i in range(idx, min(idx + allowable_delay, len(df))):
                if df.majority_voting[i] == 1:
                    delay = df.time[i] - df.time[idx]
                    contact_delays.append(delay)
                    break
            
            if i >= (idx + allowable_delay - 1):
                contact_delays.append(0)

            # Calculate TP and FN
            for j in range(change_events[event_idx+1], max(change_events[event_idx+1]-allowable_delay, 0), -1):
                if df.majority_voting[j] == 1:
                    break

            for idx_ in range(i, j):
                if df.majority_voting[idx_] == df.label[idx_]:
                    TP += 1
                    df.loc[idx_, 'TP'] = 0.9
                else:
                    FN += 1
                    df.loc[idx_, 'FN'] = 0.9

        else: # Starting a contact-free event

            for i in range(idx, min(idx + allowable_delay, len(df))):
                if df.majority_voting[i] == 0:
                    delay = df.time[i] - df.time[idx]
                    no_contact_delays.append(delay)
                    break
            
            if i >= (idx + allowable_delay - 1):
                no_contact_delays.append(0)

            # Calculate TN and FP
            for j in range(change_events[event_idx+1], max(change_events[event_idx+1]-allowable_delay, 0), -1):
                if df.majority_voting[j] == 0:
                    break
                
            for idx_ in range(i, j):
                if df.majority_voting[idx_] == df.label[idx_]:
                    TN += 1
                    df.loc[idx_, 'TN'] = 0.1
                else:
                    FP += 1
                    df.loc[idx_, 'FP'] = 0.1

    # Compute averages safely
    contact_avg_delay = np.nanmean([d for d in contact_delays if d > 0])
    no_contact_avg_delay = np.nanmean([d for d in no_contact_delays if d > 0])

    ModelAccuracy = (TP + TN) / (TP + TN + FP + FN) * 100
    DetectionFailureRate = FN / (TP + FN) * 100
    FalseAlarmRate = FP / (TN + FP) * 100

    return df, TP, TN, FP, FN, contact_avg_delay, no_contact_avg_delay, ModelAccuracy, DetectionFailureRate, FalseAlarmRate, contact_delays, no_contact_delays


def majority_voting_last_n(model_out, n):
    """
    Apply majority voting for each step considering the last `n` items.

    Args:
        model_out (pd.Series): Predicted classes for each step.
        n (int): Number of previous items (including the current one) to consider for voting.

    Returns:
        pd.Series: Smoothed predictions based on majority voting.
    """
    smoothed_predictions = model_out.copy()  # Copy to retain index
    for i in range(n, len(model_out), 1):
        # Define the window
        start_idx = i - n 
        end_idx = i  # Include the current item
        window = model_out.iloc[start_idx:end_idx]
        
        # Perform majority voting
        most_common = Counter(window).most_common(1)[0][0]
        smoothed_predictions.iloc[i-1] = most_common
    
    return smoothed_predictions

def plot_loss(loss_seq, lr_seq):
    fig, ax1 = plt.subplots(figsize=(10, 6))

    # Plot loss on the primary y-axis
    ax1.plot(loss_seq, label='Loss', color='blue', marker='o')
    ax1.set_xlabel('Epochs', fontsize=14)
    ax1.set_ylabel('Loss', fontsize=14)
    ax1.tick_params(axis='y', labelcolor='blue')
    
    # Set y-axis limits for loss to include 0
    ax1.set_ylim(bottom=0, top=max(loss_seq) * 1.05)  # Adjust top limit as needed

    ax1.grid(True)

    # Create a secondary y-axis
    ax2 = ax1.twinx()
    ax2.plot(lr_seq, label='Learning Rate', color='red', marker='o')
    ax2.set_ylabel('Learning Rate', fontsize=14)
    ax2.tick_params(axis='y', labelcolor='red')

    # Set y-axis limits for learning rate to include 0
    ax2.set_ylim(bottom=0, top=max(lr_seq) * 1.05)  # Adjust top limit as needed

    # Add legends
    lines, labels = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines + lines2, labels + labels2, loc='best')

    # Improve layout
    plt.tight_layout()

    # Show the plot
    plt.show()

# Bar chart plotting function
def plot_accuracies(pre_accuracies, post_accuracies, dataloader_names, model_name):
    labels = dataloader_names
    bar_width = 0.35
    index = range(len(labels))

    # Plotting the barchart
    plt.figure(figsize=(10, 6))
    plt.grid(True)

    bars1 = plt.bar(index, pre_accuracies, bar_width, label='Pre-DomainAdaptation Accuracy', color='blue')
    bars2 = plt.bar([i + bar_width for i in index], post_accuracies, bar_width, label='Post-DomainAdaptation Accuracy', color='green')

    # Set y-axis limits from 0 to 100
    plt.ylim(0, 120)

    # Adding labels, title, and legend
    plt.xlabel('Test Dataset', fontsize=14)
    plt.ylabel('Accuracy (%)', fontsize=14)  # Indicating accuracy as a percentage
    plt.title(f'Accuracy Comparison for {model_name}', fontsize=16)
    plt.xticks([i + bar_width / 2 for i in index], labels)
    plt.legend()

    # Adding accuracy values on top of the bars
    for bar in bars1:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width() / 2, yval + 1, f'{yval:.2f}', ha='center', fontsize=12)

    for bar in bars2:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width() / 2, yval + 1, f'{yval:.2f}', ha='center', fontsize=12)

    # Show the plot
    plt.tight_layout()
    plt.show()


In [ ]:
# hyperparameters
import pickle, logging
accuracy_metric = Accuracy()

loss_fn = nn.CrossEntropyLoss()
loss_fn = nn.BCEWithLogitsLoss()
#loss_fn = nn.MultiLabelSoftMarginLoss()

lrs = [0.04, 0.004]
lr_threshold = 0.0005
n_epochs = [35, 15]
n = 14  # Window size for majority voting
batch_size = 71
data_name , dof= 'franka_main', 7 #franka_main, ur5
split_rate = 0.75

torch.manual_seed(2020)
np.random.seed(2020)
random.seed(2020)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

if device.type == "cuda":
    print("Using GPU:", torch.cuda.get_device_name())

main_path = os.getcwd().replace('pipelines', '')
dataset_info = {os.getcwd().replace('pipelines', '') + f'/dataset/{data_name}/labeled_data/': dof}
logging.basicConfig(
    level=logging.INFO,  # Set the logging level to INFO
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(),  # Print log to console
        logging.FileHandler(os.getcwd() + f'/trained_models/{data_name}/contact_detection/{data_name}_training_log_batch_size{batch_size}_{time.time()}.txt')#num_layers{num_layers}_hidden_size{hidden_size}.txt')  # Save log to a single file
    ]
)
dict_label = {'link7': 1, 'link6':1, 'link5':1, 'link4':1, 'link3':1, 'link2':1, 'link1':1, 'no_contact': 0}
selected_features = [f'e{i}' for i in range(dof)]#+[ f'de{i}' for i in range(robot_dof)]

for num_layers in [1,2,3]:
    for hidden_size in [32, 64, 128, 256]:
        for seq_num in [30, 50, 80, 100, 150, 200]:
            for gap in [3, 5, 10, 15]:
                # Load the dataset from the file _half_0_75_0_3
                with open(f'{main_path}/dataset/{data_name}/pickleDatasets/{data_name}_feature_e_gap_{gap}_splitRate_{split_rate}_seqNum_{seq_num}.pickle', 'rb') as f:
                    master_dataset = pickle.load(f)

                # Training
                train_dataloader = DataLoader(master_dataset, batch_size=batch_size, shuffle=True)

                # Build the model
                #model_lstmBlock = lstmBlock(num_features_joints=seq_num, num_layers=num_layers, hidden_size=64, dropout=0.2, bidirectional=True)
                model_cnnLSTM = cnnLSTM(num_features_joints=seq_num, num_layers=num_layers, hidden_size=hidden_size, dropout=0.7, bidirectional=True)
                models = [model_cnnLSTM, model_cnnLSTM]
                models_names = ['model_lstmBlock',  'model_cnnLSTM',]
                for model in models:
                    model.to(device)
                    
                for ii in [1]:#range(len(models)):
                    model = models[ii]
                    #model.lstm.flatten_parameters()
                    
                    lr = lrs[ii]
                    logging.info(f'------------  {models_names[ii]} , num_layers = {num_layers}, hidden_size={hidden_size} seq_num = {seq_num}, gap = {gap}, --------------')
                    #  training on source robot
                    #model, loss_seq, lr_seq = train_loop(model, train_datasetloader,selected_features, lr, n_epochs[ii])
                    model.train()
                    # Use Adam optimizer and CrossEntropyLoss as the loss function
                    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
                    # Initialize learning rate scheduler
                    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=1, verbose=True)
                    loss_seq = []
                    lr_seq = []
                    # Training loop
                    for epoch in range(n_epochs[ii]):
                        running_loss = []
                        
                        for X_batch, y_batch in train_dataloader:
                            X_batch, y_batch = X_batch.to(device), y_batch.to(device) # Move data to device
                            optimizer.zero_grad()
                            y_pred = model(X_batch)
                            y_batch = create_hierarchical_labels(y_batch, dof)
                            loss = loss_fn(y_pred, y_batch)            
                            loss.backward()
                            optimizer.step()
                            running_loss.append(loss.cpu().detach().numpy())

                        avg_loss = np.mean(running_loss)
                        #print(f"Epoch: {epoch + 1}/{n_epochs} - learning rate: {optimizer.param_groups[0]['lr']:.5f}, classification loss: {avg_loss:.4f}")
                        # Update the scheduler with the average loss
                        scheduler.step(avg_loss)
                        current_lr = optimizer.param_groups[0]['lr']
                        if current_lr < lr_threshold:
                            print(f"Learning rate has dropped below the threshold of {lr_threshold}. Stopping training.")
                            break
                        loss_seq.append(avg_loss)
                        lr_seq.append(current_lr)
                        if avg_loss < 0.08:
                            print('early stopping <0.08!')
                            break
                    #logging.info(f'loss = {loss_seq}')
                    #logging.info(f'lr = {lr_seq}')
                    model.eval()
                    os.makedirs(f'{main_path}/pipelines/trained_models/{data_name}/contact_detection/{batch_size}/', exist_ok=True)
                    accuracies = []
                    # Loop through each dataset path and corresponding dof
                    for data_path, dof in dataset_info.items():
                        # Load the dataset for testing
                        testing_datasets = LoadDatasets(data_path, dict_label)
                        test_datasetloader = DataLoader(testing_datasets, batch_size=1, shuffle=False)
                        
                        for trial_dataset_path, label in test_datasetloader:
                            if 'link1' in trial_dataset_path[0]:
                                # Load the dataset for the specific trial
                                data = LoadSeqDataset(file_path=trial_dataset_path[0], label=label[0], 
                                                    selected_features=selected_features[0:dof], seq_num=seq_num, gap=1)
                                data.sequences = data.sequences[(len(data)-len(data)//2):len(data)]
                                # Initialize an empty dataframe if not done before
                                df = pd.DataFrame(columns=["time", "label", "model_out", "probability", "majority_voting"])

                                # Create a DataLoader for this specific trial data
                                test_loader = DataLoader(data, batch_size=len(data), shuffle=False)

                                # Iterate through the DataLoader to make predictions
                                for batch_idx, (seqs, labels) in enumerate(test_loader):
                                    seqs = seqs.float().to(device)  # Convert sequences to float and move to device

                                    # Perform the prediction without gradient computation
                                    with torch.no_grad():
                                        predictions = model.prediction(seqs)

                                    # Fill the dataframe with the results
                                    df['time'] = data.data.time[len(data.data.time)-len(data)-1:-1]
                                    df['label'] = labels.cpu().numpy()  # Convert label to numpy
                                    df['model_out'] = predictions.cpu().detach().numpy()  # Convert predictions to numpy
                                    df['majority_voting'] = majority_voting_last_n(df['model_out'], n)

                                # Call the function
                                df, TP, TN, FP, FN, contact_avg_delay, no_contact_avg_delay, ModelAccuracy, DetectionFailureRate, FalseAlarmRate, contact_delays, no_contact_delays = contact_detection_accuracy(df)
                                print('contact_delays:', contact_delays)
                                print('no_contact_delays:', no_contact_delays)
                                logging.info(f"Accuracy:{ModelAccuracy:.2f}, DetectionFailurRate:{DetectionFailureRate:.2f}, FalseAlarm:{FalseAlarmRate:.2f}, Contact Detection Delay: {contact_avg_delay*1000:.3f}ms, No-Contact Detection Delay: {no_contact_avg_delay*1000:.3f}ms, TP: {TP}, TN: {TN}, FP: {FP}, FN: {FN}")

                                df[['time','label', 'majority_voting', 'TP', 'TN', 'FP', 'FN']].iplot( x='time',kind='scatter', 
                                    mode={'label': 'lines', 'majority_voting': 'lines', 'TP': 'markers', 'TN': 'markers', 'FP': 'markers', 'FN': 'markers', },
                                    title='Contact Detection Analysis (Interactive)',
                                    xTitle='Time Index',yTitle='Signal',theme='white',
                                    symbol=[None, None, 'x','x', 'x', 'x'],
                                    colors=['blue', 'pink', 'green', 'lightgreen', 'orange', 'red'],
                                )
                                break
                        torch.save(model.state_dict(), f'{main_path}/pipelines/trained_models/{data_name}/contact_detection/{batch_size}/numLayer{num_layers}_hiddenSize{hidden_size}_seq_num{seq_num}_gap{gap}_accuracy{ModelAccuracy:.2f}.pth')


Using GPU: Quadro RTX 8000


2025-09-09 10:53:09,484 - INFO - ------------  model_cnnLSTM , num_layers = 1, hidden_size=32 seq_num = 30, gap = 3, --------------
/home/rzma/miniconda3/envs/frankapyenv/lib/python3.6/site-packages/ipykernel_launcher.py:110: RuntimeWarning:

Mean of empty slice

2025-09-09 10:56:37,678 - INFO - Accuracy:79.52, DetectionFailurRate:71.15, FalseAlarm:0.00, Contact Detection Delay: 97.507ms, No-Contact Detection Delay: nanms, TP: 73, TN: 626, FP: 0, FN: 180


contact_delays: [0.089998245, 0, 0, 0.109986305, 0.09997654, 0.094997406, 0.09504223, 0.095044136]
no_contact_delays: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


## Accuracy calculation on all links

In [25]:
data_name, dof = 'franka_main', 7
batch_size = 70

best_models_index, folder_name, data_name, dof = ([ [1, 64, 3, 50 ],
                                                    [1, 64, 3, 80 ],
                                                    [1, 64, 3, 100 ],                                                    
                                                    [1, 64, 5, 50 ],
                                                    [1, 64, 5, 80 ],
                                                    [1, 64, 5, 100 ],
                                                    [1, 128, 3, 50 ],
                                                    [1, 128, 3, 80 ],
                                                    [1, 128, 3, 100 ],
                                                    [1, 128, 5, 50 ],
                                                    [1, 128, 5, 80 ],
                                                    [1, 128, 5, 100 ],
                                                        ], f'pipelines/trained_models/franka_main/contact_detection/{batch_size}', f'{data_name}', dof)
dataset_info = {
os.getcwd().replace('pipelines', '') + f'/dataset/{data_name}/labeled_data/': dof
}


# testing all models
best_models_index=[]
for num_layers in [1, 2, 3]:
    for hidden_size in [32, 64, 128, 256]:
        for seq_num in [30, 50, 80, 100, 150, 200]:
            for gap in [3, 5, 10, 15]:
                best_models_index.append([num_layers, hidden_size, gap, seq_num])

# hyperparameters

n = 14  # Window size for majority voting

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

if device.type == "cuda":
    print("Using GPU:", torch.cuda.get_device_name())

main_path = os.getcwd().replace('pipelines', '')


logging.info(f'trained on {folder_name}, test on {data_name}')
dict_label = {'link7': 1, 'link6':1, 'link5':1, 'link4':1, 'link3':1, 'link2':1, 'link1':1, 'no_contact': 0}
selected_features = [f'e{i}' for i in range(dof)]#+[ f'de{i}' for i in range(robot_dof)]


counter = 1
links = ['link7', 'link6', 'link5', 'link4', 'link3', 'link2', 'link1']
links_label = {'link7':'a6', 'link6':'b9', 'link5':'c12', 'link4':'d5', 'link3':'e4', 'link2':'f5', 'link1':'g1'}


for num_layers, hidden_size, gap, seq_num in best_models_index:
    for model_name in os.listdir(f'{main_path}/{folder_name}/'):
        if f'numLayer{num_layers}_hiddenSize{hidden_size}_seq_num{seq_num}_gap{gap}' in model_name:
            counter +=1
            model = cnnLSTM(num_features_joints=seq_num, num_layers=num_layers, hidden_size=hidden_size, dropout=0.7, bidirectional=True)
            model.to(device)
            model.load_state_dict(torch.load(f'{main_path}/{folder_name}/{model_name}'))
            model.eval()
            y_pred, y_true= [], []

            for data_path, dof in dataset_info.items():
                # Load the dataset for testing
                testing_datasets = LoadDatasets(data_path, dict_label)
                test_datasetloader = DataLoader(testing_datasets, batch_size=1, shuffle=False)
                TN_t, TP_t, FP_t, FN_t = 0,0,0,0
                contact_avg_delay_t, no_contact_avg_delay_t=[],[]

                for link in links:
                    # Iterate through the dataset (trials)
                    for trial_dataset_path, label in test_datasetloader:
                        if link in trial_dataset_path[0]:
                            if data_name== 'franka_main' and links_label[link] not in trial_dataset_path[0]:
                                pass
                            else:
                                # Load the dataset for the specific trial
                                data = LoadSeqDataset(file_path=trial_dataset_path[0], label=label[0], 
                                                    selected_features=selected_features[0:dof], seq_num=seq_num, gap=1)
                                data.sequences = data.sequences[(len(data)-len(data)//2):-1]
                                # Initialize an empty dataframe if not done before
                                df = pd.DataFrame(columns=["time", "label", "model_out", "probability", "majority_voting"])

                                # Create a DataLoader for this specific trial data
                                test_loader = DataLoader(data, batch_size=len(data), shuffle=False)

                                # Iterate through the DataLoader to make predictions
                                for batch_idx, (seqs, labels) in enumerate(test_loader):
                                    seqs = seqs.float().to(device)  # Convert sequences to float and move to device

                                    # Perform the prediction without gradient computation
                                    with torch.no_grad():
                                        predictions = model.prediction(seqs)

                                    # Fill the dataframe with the results
                                    df['time'] = data.data.time[len(data.data.time)-len(data)-1:-1]
                                    df['label'] = labels.cpu().numpy()  # Convert label to numpy
                                    df['model_out'] = predictions.cpu().detach().numpy()  # Convert predictions to numpy
                                    df['majority_voting'] = majority_voting_last_n(df['model_out'], n)

                                    df, TP, TN, FP, FN, contact_avg_delay, no_contact_avg_delay, ModelAccuracy, DetectionFailureRate, FalseAlarmRate, contact_delays, no_contact_delays = contact_detection_accuracy(df)
                                    '''print('contact_delays:', contact_delays)
                                    print('no_contact_delays:', no_contact_delays)
                                    print(f"Accuracy:{ModelAccuracy:.2f}, DetectionFailurRate:{DetectionFailureRate:.2f}, FalseAlarm:{FalseAlarmRate:.2f}, Contact Detection Delay: {contact_avg_delay*1000:.3f}ms, No-Contact Detection Delay: {no_contact_avg_delay*1000:.3f}ms, TP: {TP}, TN: {TN}, FP: {FP}, FN: {FN}")

                                    df[['time','label', 'majority_voting', 'TP', 'TN', 'FP', 'FN']].iplot( x='time',kind='scatter', 
                                        mode={'label': 'lines', 'majority_voting': 'lines', 'TP': 'markers', 'TN': 'markers', 'FP': 'markers', 'FN': 'markers', },
                                        title=f'Contact Detection Analysis ({link})',
                                        xTitle='Time Index',yTitle='Signal',theme='white',
                                        symbol=[None, None, 'x','x', 'x', 'x'],
                                        colors=['blue', 'pink', 'green', 'lightgreen', 'orange', 'red'],
                                    )'''
                                    TP_t, TN_t, FP_t, FN_t = TP_t + TP, TN_t + TN, FP_t + FP, FN_t + FN
                                    contact_avg_delay_t.append(contact_avg_delay)
                                    no_contact_avg_delay_t.append(no_contact_avg_delay)
                                break
                filtered_contact_avg = [x for x in contact_avg_delay_t if x != 0 and not np.isnan(x)]
                filtered_no_contact_avg = [x for x in no_contact_avg_delay_t if x != 0 and not np.isnan(x)]

                contact_avg_delay_t = np.mean(filtered_contact_avg)
                no_contact_avg_delay_t = np.mean(filtered_no_contact_avg)
                logging.info(f"Testing: {num_layers}, {hidden_size}, {gap}, {seq_num}, Accuracy:{(TP_t+TN_t)/(TP_t+TN_t+FP_t+FN_t)*100:.2f}, DetectionFailurRate:{FN_t/(FN_t+TP_t)*100:.2f}, FalseAlarm:{FP_t/(FP_t+TN_t)*100:.2f}, Contact Detection Delay: {contact_avg_delay_t*1000:.2f}ms, No-Contact Detection Delay: {no_contact_avg_delay_t*1000:.2f}ms, TP: {TP_t}, TN: {TN_t}, FP: {FP_t}, FN: {FN_t}")


2025-09-09 12:22:26,127 - INFO - trained on pipelines/trained_models/franka_main/contact_detection/70, test on franka_main


Using GPU: Quadro RTX 8000


/home/rzma/miniconda3/envs/frankapyenv/lib/python3.6/site-packages/ipykernel_launcher.py:110: RuntimeWarning:

Mean of empty slice

/home/rzma/miniconda3/envs/frankapyenv/lib/python3.6/site-packages/ipykernel_launcher.py:109: RuntimeWarning:

Mean of empty slice

/home/rzma/miniconda3/envs/frankapyenv/lib/python3.6/site-packages/ipykernel_launcher.py:110: RuntimeWarning:

Mean of empty slice

/home/rzma/miniconda3/envs/frankapyenv/lib/python3.6/site-packages/ipykernel_launcher.py:110: RuntimeWarning:

Mean of empty slice

2025-09-09 12:22:41,519 - INFO - Testing: 32, 3, 30, Accuracy:90.02, DetectionFailurRate:16.03, FalseAlarm:5.00, Contact Detection Delay: 72.18ms, No-Contact Detection Delay: 100.12ms, TP: 3694, TN: 5087, FP: 268, FN: 705
2025-09-09 12:22:56,803 - INFO - Testing: 32, 3, 30, Accuracy:90.16, DetectionFailurRate:16.34, FalseAlarm:3.54, Contact Detection Delay: 68.30ms, No-Contact Detection Delay: 71.48ms, TP: 4254, TN: 5069, FP: 186, FN: 831
/home/rzma/miniconda3/envs/fr

# 4. Transfer Learning

## Fine Tuning

In [ ]:
import pickle, logging
data_name, dof = 'franka_mindlab', 7
best_models_index, folder_name, data_name,batch_size, dof = ([   [1, 64, 3, 50 ],
                                                                  [1, 64, 3, 80 ],
                                                                  [1, 64, 3, 100 ],
                                                                  [1, 128, 3, 50 ],
                                                                  [1, 128, 3, 80 ],
                                                                  [1, 128, 3, 100 ],
                                                                  [1, 64, 5, 50 ],
                                                                  [1, 64, 5, 80 ],
                                                                  [1, 64, 5, 100 ],
                                                                  [1, 128, 5, 50 ],
                                                                  [1, 128, 5, 80 ],
                                                                  [1, 128, 5, 100 ],
                                                                      ], 'pipelines/trained_models/franka_main/contact_detection/63', f'{data_name}',71, dof)


lrs = [0.04, 0.001]
lr_threshold = 0.0005
n_epochs = [35, 20]
n = 14  # Window size for majority voting
logging.info(f'Fine-tuning from {folder_name} on {data_name}')
torch.manual_seed(2020)
np.random.seed(2020)
random.seed(2020)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

if device.type == "cuda":
    print("Using GPU:", torch.cuda.get_device_name())

main_path = os.getcwd().replace('pipelines', '')
dataset_info = {os.getcwd().replace('pipelines', '') + f'/dataset/{data_name}/labeled_data/': dof}

#dict_label = {'link7': 7, 'link6':6, 'link5':5, 'link4':4, 'link3':3, 'link2':2, 'link1':1, 'no_contact': 0}
selected_features = [f'e{i}' for i in range(dof)]#+[ f'de{i}' for i in range(robot_dof)]

links = ['link7', 'link6', 'link5', 'link4', 'link3', 'link2', 'link1']


for num_layers, hidden_size, gap, seq_num in best_models_index:
    #for gap in [3]:#, 5]:#, 10, 15]:
        # Load the dataset from the file
        with open(f'{main_path}/dataset/{data_name}/pickleDatasets/{data_name}_feature_e_gap_{gap}_splitRate_{split_rate}_seqNum_{seq_num}.pickle', 'rb') as f:
            master_dataset = pickle.load(f)
    
        # Training
        train_dataloader = DataLoader(master_dataset, batch_size=batch_size, shuffle=True)

        for model_name in os.listdir(f'{main_path}/{folder_name}/'):
            if f'hiddenSize{hidden_size}_seq_num{seq_num}_gap{gap}' in model_name:
                # Build the model
                print(model_name)
                model = cnnLSTM(num_features_joints=seq_num, num_layers=num_layers, hidden_size=hidden_size, dropout=0.7, bidirectional=True)
                model.to(device)
                model.load_state_dict(torch.load(f'{main_path}/{folder_name}/{model_name}'))
                model.train()
                    
                lr = lrs[1]
                #print(f'------------  {models_names[ii]} , num_layers = {num_layers}, hidden_size={hidden_size} seq_num = {seq_num}, gap = {gap}, --------------')
                optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
                scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=1, verbose=True)
                loss_seq = []
                lr_seq = []
                # Training loop
                for epoch in range(n_epochs[1]):
                    running_loss = []
                    
                    for X_batch, y_batch in train_dataloader:
                        X_batch, y_batch = X_batch.to(device), y_batch.to(device) # Move data to device
                        optimizer.zero_grad()
                        y_pred = model(X_batch)
                        y_batch = create_hierarchical_labels(y_batch, dof)
                        loss = loss_fn(y_pred, y_batch)            
                        loss.backward()
                        optimizer.step()
                        running_loss.append(loss.cpu().detach().numpy())

                    avg_loss = np.mean(running_loss)
                    print(f"Epoch: {epoch + 1}/{n_epochs} - learning rate: {optimizer.param_groups[0]['lr']:.5f}, classification loss: {avg_loss:.4f}")
                    # Update the scheduler with the average loss
                    scheduler.step(avg_loss)
                    current_lr = optimizer.param_groups[0]['lr']
                    if current_lr < lr_threshold:
                        print(f"Learning rate has dropped below the threshold of {lr_threshold}. Stopping training.")
                        break
                    loss_seq.append(avg_loss)
                    lr_seq.append(current_lr)
                    if avg_loss < 0.085:
                        print('early stopping <0.47!')
                        break
                #logging.info(f'loss = {loss_seq}')
                #logging.info(f'lr = {lr_seq}')
                model.eval()
                os.makedirs(f'{main_path}/pipelines/trained_models/{data_name}/contact_detection/fine_tuned{batch_size}/', exist_ok=True)
                accuracies = []
                # Loop through each dataset path and corresponding dof
                for data_path, dof in dataset_info.items():
                    # Load the dataset for testing
                    testing_datasets = LoadDatasets(data_path, dict_label)
                    test_datasetloader = DataLoader(testing_datasets, batch_size=1, shuffle=False)
                    
                    for trial_dataset_path, label in test_datasetloader:
                        if 'link1' in trial_dataset_path[0]:
                            # Load the dataset for the specific trial
                            data = LoadSeqDataset(file_path=trial_dataset_path[0], label=label[0], 
                                                selected_features=selected_features[0:dof], seq_num=seq_num, gap=1)
                            data.sequences = data.sequences[(len(data)-len(data)//2):len(data)]
                            # Initialize an empty dataframe if not done before
                            df = pd.DataFrame(columns=["time", "label", "model_out", "probability", "majority_voting"])

                            # Create a DataLoader for this specific trial data
                            test_loader = DataLoader(data, batch_size=len(data), shuffle=False)

                            # Iterate through the DataLoader to make predictions
                            for batch_idx, (seqs, labels) in enumerate(test_loader):
                                seqs = seqs.float().to(device)  # Convert sequences to float and move to device

                                # Perform the prediction without gradient computation
                                with torch.no_grad():
                                    predictions = model.prediction(seqs)

                                # Fill the dataframe with the results
                                df['time'] = data.data.time[len(data.data.time)-len(data)-1:-1]
                                df['label'] = labels.cpu().numpy()  # Convert label to numpy
                                df['model_out'] = predictions.cpu().detach().numpy()  # Convert predictions to numpy
                                df['majority_voting'] = majority_voting_last_n(df['model_out'], n)

                            df, TP, TN, FP, FN, contact_avg_delay, no_contact_avg_delay, ModelAccuracy, DetectionFailureRate, FalseAlarmRate, contact_delays, no_contact_delays = contact_detection_accuracy(df)
                            print('contact_delays:', contact_delays)
                            print('no_contact_delays:', no_contact_delays)
                            logging.info(f"Testing: {hidden_size}, {gap}, {seq_num}, Accuracy:{ModelAccuracy:.2f}, DetectionFailurRate:{DetectionFailureRate:.2f}, FalseAlarm:{FalseAlarmRate:.2f}, Contact Detection Delay: {contact_avg_delay*1000:.3f}ms, No-Contact Detection Delay: {no_contact_avg_delay*1000:.3f}ms, TP: {TP}, TN: {TN}, FP: {FP}, FN: {FN}")

                            df[['time','label', 'majority_voting', 'TP', 'TN', 'FP', 'FN']].iplot( x='time',kind='scatter', 
                                mode={'label': 'lines', 'majority_voting': 'lines', 'TP': 'markers', 'TN': 'markers', 'FP': 'markers', 'FN': 'markers', },
                                title=f'Contact Detection, {trial_dataset_path[0]}',
                                xTitle='Time Index',yTitle='Signal',theme='white',
                                symbol=[None, None, 'x','x', 'x', 'x'],
                                colors=['blue', 'pink', 'green', 'lightgreen', 'orange', 'red'],
                            )
                            break
                    torch.save(model.state_dict(), f'{main_path}/pipelines/trained_models/{data_name}/contact_detection/fine_tuned{batch_size}/hiddenSize{hidden_size}_seq_num{seq_num}_gap{gap}_accuracy{ModelAccuracy:.2f}.pth')
                    break 

2025-09-09 11:05:21,073 - INFO - Fine-tuning from pipelines/trained_models/franka_main/contact_detection/63 on franka_mindlab


Using GPU: Quadro RTX 8000
numLayer1_hiddenSize64_seq_num80_gap3_accuracy98.76
Epoch: 1/[35, 20] - learning rate: 0.00100, classification loss: 0.4250
Epoch: 2/[35, 20] - learning rate: 0.00100, classification loss: 0.2991
Epoch: 3/[35, 20] - learning rate: 0.00100, classification loss: 0.2549
Epoch: 4/[35, 20] - learning rate: 0.00100, classification loss: 0.2187
Epoch: 5/[35, 20] - learning rate: 0.00100, classification loss: 0.1876
Epoch: 6/[35, 20] - learning rate: 0.00100, classification loss: 0.1678
Epoch: 7/[35, 20] - learning rate: 0.00100, classification loss: 0.1415
Epoch: 8/[35, 20] - learning rate: 0.00100, classification loss: 0.1227
Epoch: 9/[35, 20] - learning rate: 0.00100, classification loss: 0.1013
Epoch: 10/[35, 20] - learning rate: 0.00100, classification loss: 0.1020
Epoch: 11/[35, 20] - learning rate: 0.00100, classification loss: 0.0835
early stopping <0.47!


2025-09-09 11:05:31,827 - INFO - Testing: 64, 3, 80, Accuracy:85.01, DetectionFailurRate:19.30, FalseAlarm:10.88, Contact Detection Delay: 67.524ms, No-Contact Detection Delay: 91.995ms, TP: 694, TN: 803, FP: 98, FN: 166


contact_delays: [0.0, 0.14003944, 0.024967194, 0.030031204, 0.0, 0.049975395, 0.07008171, 0.09004784]
no_contact_delays: [0, 0.09000206, 0.115000725, 0, 0.050020218, 0.124975204, 0, 0.07997513]


### Accuracy Calculation on all links


In [16]:
#data_name , dof = 'ur5', 6
best_models_index, folder_name, data_name, dof = ([ [1, 64, 3, 50 ],
                                                    [1, 64, 3, 80 ],
                                                    [1, 64, 3, 100 ],
                                                    [1, 128, 3, 50 ],
                                                    [1, 128, 3, 80 ],
                                                    [1, 128, 3, 100 ],
                                                    [1, 64, 5, 50 ],
                                                    [1, 64, 5, 80 ],
                                                    [1, 64, 5, 100 ],
                                                    [1, 128, 5, 50 ],
                                                    [1, 128, 5, 80 ],
                                                    [1, 128, 5, 100 ],
                                                        ],  f'pipelines/trained_models/{data_name}/contact_detection/fine_tuned{batch_size}', f'{data_name}', dof)

dataset_info = {
os.getcwd().replace('pipelines', '') + f'/dataset/{data_name}/labeled_data/': dof
}

# hyperparameters

n = 14  # Window size for majority voting

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

if device.type == "cuda":
    print("Using GPU:", torch.cuda.get_device_name())

main_path = os.getcwd().replace('pipelines', '')

logging.info(f'Fine- tuned on {folder_name}, test on {data_name}')
dict_label = {'link7': 1, 'link6':1, 'link5':1, 'link4':1, 'link3':1, 'link2':1, 'link1':1, 'no_contact': 0}
selected_features = [f'e{i}' for i in range(dof)]#+[ f'de{i}' for i in range(robot_dof)]

counter = 1
links = ['link7', 'link6', 'link5', 'link4', 'link3', 'link2', 'link1']
for num_layers, hidden_size, gap, seq_num in best_models_index:
    for model_name in os.listdir(f'{main_path}/{folder_name}/'):
        if f'hiddenSize{hidden_size}_seq_num{seq_num}_gap{gap}' in model_name:
            counter +=1
            model = cnnLSTM(num_features_joints=seq_num, num_layers=num_layers, hidden_size=hidden_size, dropout=0.7, bidirectional=True)
            model.to(device)
            model.load_state_dict(torch.load(f'{main_path}/{folder_name}/{model_name}'))
            model.eval()
            y_pred, y_true= [], []

            for data_path, dof in dataset_info.items():
                # Load the dataset for testing
                testing_datasets = LoadDatasets(data_path, dict_label)
                test_datasetloader = DataLoader(testing_datasets, batch_size=1, shuffle=False)
                TN_t, TP_t, FP_t, FN_t = 0,0,0,0
                contact_avg_delay_t, no_contact_avg_delay_t=[],[]

                for link in links:
                    # Iterate through the dataset (trials)
                    for trial_dataset_path, label in test_datasetloader:
                        if link in trial_dataset_path[0]:
                            # Load the dataset for the specific trial
                            data = LoadSeqDataset(file_path=trial_dataset_path[0], label=label[0], 
                                                selected_features=selected_features[0:dof], seq_num=seq_num, gap=1)
                            data.sequences = data.sequences[(len(data)-len(data)//2):-1]
                            # Initialize an empty dataframe if not done before
                            df = pd.DataFrame(columns=["time", "label", "model_out", "probability", "majority_voting"])

                            # Create a DataLoader for this specific trial data
                            test_loader = DataLoader(data, batch_size=len(data), shuffle=False)

                            # Iterate through the DataLoader to make predictions
                            for batch_idx, (seqs, labels) in enumerate(test_loader):
                                seqs = seqs.float().to(device)  # Convert sequences to float and move to device

                                # Perform the prediction without gradient computation
                                with torch.no_grad():
                                    predictions = model.prediction(seqs)

                                # Fill the dataframe with the results
                                df['time'] = data.data.time[len(data.data.time)-len(data)-1:-1]
                                df['label'] = labels.cpu().numpy()  # Convert label to numpy
                                df['model_out'] = predictions.cpu().detach().numpy()  # Convert predictions to numpy
                                df['majority_voting'] = majority_voting_last_n(df['model_out'], n)

                                df, TP, TN, FP, FN, contact_avg_delay, no_contact_avg_delay, ModelAccuracy, DetectionFailureRate, FalseAlarmRate, contact_delays, no_contact_delays = contact_detection_accuracy(df)
                                '''print('contact_delays:', contact_delays)
                                print('no_contact_delays:', no_contact_delays)
                                print(f"Accuracy:{ModelAccuracy:.2f}, DetectionFailurRate:{DetectionFailureRate:.2f}, FalseAlarm:{FalseAlarmRate:.2f}, Contact Detection Delay: {contact_avg_delay*1000:.3f}ms, No-Contact Detection Delay: {no_contact_avg_delay*1000:.3f}ms, TP: {TP}, TN: {TN}, FP: {FP}, FN: {FN}")

                                df[['time','label', 'majority_voting', 'TP', 'TN', 'FP', 'FN']].iplot( x='time',kind='scatter', 
                                    mode={'label': 'lines', 'majority_voting': 'lines', 'TP': 'markers', 'TN': 'markers', 'FP': 'markers', 'FN': 'markers', },
                                    title=f'Contact Detection Analysis ({link})',
                                    xTitle='Time Index',yTitle='Signal',theme='white',
                                    symbol=[None, None, 'x','x', 'x', 'x'],
                                    colors=['blue', 'pink', 'green', 'lightgreen', 'orange', 'red'],
                                )'''
                                TP_t, TN_t, FP_t, FN_t = TP_t + TP, TN_t + TN, FP_t + FP, FN_t + FN
                                contact_avg_delay_t.append(contact_avg_delay)
                                no_contact_avg_delay_t.append(no_contact_avg_delay)
                            break
                filtered_contact_avg = [x for x in contact_avg_delay_t if x != 0 and not np.isnan(x)]
                filtered_no_contact_avg = [x for x in no_contact_avg_delay_t if x != 0 and not np.isnan(x)]

                contact_avg_delay_t = np.mean(filtered_contact_avg)
                no_contact_avg_delay_t = np.mean(filtered_no_contact_avg)
                logging.info(f"Testing: {num_layers}, {hidden_size}, {gap}, {seq_num}, Accuracy:{(TP_t+TN_t)/(TP_t+TN_t+FP_t+FN_t)*100:.2f}, DetectionFailurRate:{FN_t/(FN_t+TP_t)*100:.2f}, FalseAlarm:{FP_t/(FP_t+TN_t)*100:.2f}, Contact Detection Delay: {contact_avg_delay_t*1000:.2f}ms, No-Contact Detection Delay: {no_contact_avg_delay_t*1000:.2f}ms, TP: {TP_t}, TN: {TN_t}, FP: {FP_t}, FN: {FN_t}")

2025-09-09 11:07:41,685 - INFO - Fine- tuned on pipelines/trained_models/franka_mindlab/contact_detection/fine_tuned71, test on franka_mindlab


Using GPU: Quadro RTX 8000


2025-09-09 11:07:58,861 - INFO - Testing: 64, 3, 80, Accuracy:92.21, DetectionFailurRate:6.12, FalseAlarm:9.00, Contact Detection Delay: 67.98ms, No-Contact Detection Delay: 47.26ms, TP: 4905, TN: 6499, FP: 643, FN: 320
